# Tugas Praktik — Building Batch Data Pipelines with Spark
## Studi kasus: Pipeline penjualan e-commerce

**Tujuan:** melengkapi kode PySpark untuk membangun batch data pipeline dari data transaksi mentah menjadi data yang telah dibersihkan dan diringkas.

### Skenario bisnis
Tim bisnis ingin mendapatkan ringkasan penjualan per kategori produk. Data mentah dapat mengandung nilai kosong, duplikasi, atau transaksi yang tidak valid. Tugas peserta adalah membaca data, melakukan validasi dan transformasi, membuat agregasi, lalu menyimpan output ke Parquet.

### Petunjuk pengerjaan
1. Lengkapi seluruh bagian `# TODO`.
2. Jalankan cell secara berurutan.
3. Periksa hasil setiap tahap pipeline.
4. Jawab pertanyaan refleksi pada bagian akhir.

**Catatan:** notebook ini menggunakan data contoh dan dapat dijalankan secara lokal tanpa Kafka atau API key.

## 1. Setup Spark

In [1]:
# TODO 1: import SparkSession
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Tugas_Building_Batch_Data_Pipelines")
    .master("local[*]")
    .getOrCreate()
)

print("Spark version:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/09 12:19:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


26/11/09 12:19:45 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark version: 4.2.0


## 2. Membuat data transaksi mentah

Data ini meniru file transaksi yang diterima pipeline setiap hari.

In [2]:
data = [
    ("T001", "2026-10-20", "Laptop", "Electronics", 1, 1200.0, "PAID"),
    ("T002", "2026-10-20", "Mouse", "Accessories", 2, 25.0, "PAID"),
    ("T003", "2026-10-20", "Keyboard", "Accessories", 1, 45.0, "PAID"),
    ("T004", "2026-10-20", "Monitor", "Electronics", 1, 280.0, "CANCELLED"),
    ("T005", "2026-10-21", "Laptop", "Electronics", 1, 1200.0, "PAID"),
    ("T006", "2026-10-21", "Headset", "Accessories", 3, 40.0, "PAID"),
    ("T007", "2026-10-21", "Webcam", "Accessories", 1, 60.0, "PAID"),
    ("T007", "2026-10-21", "Webcam", "Accessories", 1, 60.0, "PAID"),
    ("T008", "2026-10-21", "Tablet", "Electronics", 0, 350.0, "PAID"),
    ("T009", "2026-10-21", "Dock", "Accessories", 2, None, "PAID"),
    ("T010", "2026-10-22", "Phone", "Electronics", 2, 500.0, "PAID"),
]

columns = [
    "transaction_id", "transaction_date", "product",
    "category", "quantity", "unit_price", "status"
]

# TODO 2: buat Spark DataFrame bernama raw_df
raw_df = spark.createDataFrame(data, columns)

raw_df.show(truncate=False)

+--------------+----------------+--------+-----------+--------+----------+---------+
|transaction_id|transaction_date|product |category   |quantity|unit_price|status   |
+--------------+----------------+--------+-----------+--------+----------+---------+
|T001          |2026-10-20      |Laptop  |Electronics|1       |1200.0    |PAID     |
|T002          |2026-10-20      |Mouse   |Accessories|2       |25.0      |PAID     |
|T003          |2026-10-20      |Keyboard|Accessories|1       |45.0      |PAID     |
|T004          |2026-10-20      |Monitor |Electronics|1       |280.0     |CANCELLED|
|T005          |2026-10-21      |Laptop  |Electronics|1       |1200.0    |PAID     |
|T006          |2026-10-21      |Headset |Accessories|3       |40.0      |PAID     |
|T007          |2026-10-21      |Webcam  |Accessories|1       |60.0      |PAID     |
|T007          |2026-10-21      |Webcam  |Accessories|1       |60.0      |PAID     |
|T008          |2026-10-21      |Tablet  |Electronics|0       |35

## 3. Tahap Extract — membaca data

Dalam pipeline produksi, data biasanya dibaca dari file CSV, Parquet, database, atau data lake. Untuk tugas ini, `raw_df` berperan sebagai data mentah yang baru diterima.

In [3]:
# TODO 3: tampilkan schema dan jumlah baris data mentah
raw_df.printSchema()
print("Jumlah baris mentah:", raw_df.count())

root
 |-- transaction_id: string (nullable = true)
 |-- transaction_date: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- status: string (nullable = true)



Jumlah baris mentah: 11


## 4. Tahap Transform — menghapus duplikasi

Tugas: hapus baris transaksi duplikat berdasarkan `transaction_id`.

In [4]:
# TODO 4: deduplikasi berdasarkan transaction_id
dedup_df = raw_df.dropDuplicates(["transaction_id"])

print("Jumlah baris setelah deduplikasi:", dedup_df.count())
dedup_df.show(truncate=False)

Jumlah baris setelah deduplikasi: 10


+--------------+----------------+--------+-----------+--------+----------+---------+
|transaction_id|transaction_date|product |category   |quantity|unit_price|status   |
+--------------+----------------+--------+-----------+--------+----------+---------+
|T001          |2026-10-20      |Laptop  |Electronics|1       |1200.0    |PAID     |
|T002          |2026-10-20      |Mouse   |Accessories|2       |25.0      |PAID     |
|T003          |2026-10-20      |Keyboard|Accessories|1       |45.0      |PAID     |
|T004          |2026-10-20      |Monitor |Electronics|1       |280.0     |CANCELLED|
|T005          |2026-10-21      |Laptop  |Electronics|1       |1200.0    |PAID     |
|T006          |2026-10-21      |Headset |Accessories|3       |40.0      |PAID     |
|T007          |2026-10-21      |Webcam  |Accessories|1       |60.0      |PAID     |
|T008          |2026-10-21      |Tablet  |Electronics|0       |350.0     |PAID     |
|T009          |2026-10-21      |Dock    |Accessories|2       |NU

## 5. Validasi data

Aturan bisnis:
- `transaction_id` tidak boleh kosong.
- `quantity` harus lebih besar dari 0.
- `unit_price` tidak boleh kosong dan harus lebih besar dari atau sama dengan 0.
- Hanya transaksi berstatus `PAID` yang dihitung sebagai penjualan.

Lengkapi filter agar hanya record yang memenuhi aturan tersebut yang dipertahankan.

In [5]:
from pyspark.sql.functions import col

# TODO 5: terapkan semua aturan validasi di atas
valid_df = (
    dedup_df
    .filter(col("transaction_id").isNotNull())
    .filter(col("quantity") > 0)
    .filter(col("unit_price").isNotNull())
    .filter(col("unit_price") >= 0)
    .filter(col("status") == "PAID")
)

print("Jumlah transaksi valid:", valid_df.count())
valid_df.show(truncate=False)

Jumlah transaksi valid: 7


+--------------+----------------+--------+-----------+--------+----------+------+
|transaction_id|transaction_date|product |category   |quantity|unit_price|status|
+--------------+----------------+--------+-----------+--------+----------+------+
|T001          |2026-10-20      |Laptop  |Electronics|1       |1200.0    |PAID  |
|T002          |2026-10-20      |Mouse   |Accessories|2       |25.0      |PAID  |
|T003          |2026-10-20      |Keyboard|Accessories|1       |45.0      |PAID  |
|T005          |2026-10-21      |Laptop  |Electronics|1       |1200.0    |PAID  |
|T006          |2026-10-21      |Headset |Accessories|3       |40.0      |PAID  |
|T007          |2026-10-21      |Webcam  |Accessories|1       |60.0      |PAID  |
|T010          |2026-10-22      |Phone   |Electronics|2       |500.0     |PAID  |
+--------------+----------------+--------+-----------+--------+----------+------+



## 6. Tahap Transform — menghitung nilai transaksi

Tambahkan kolom `revenue` dengan rumus:

`revenue = quantity * unit_price`

In [6]:
# TODO 6: import withColumn sudah tersedia melalui DataFrame API; buat kolom revenue
from pyspark.sql.functions import col

sales_df = valid_df.withColumn(
    "revenue",
    col("quantity") * col("unit_price")
)

sales_df.select("transaction_id", "product", "quantity", "unit_price", "revenue").show()

+--------------+--------+--------+----------+-------+
|transaction_id| product|quantity|unit_price|revenue|
+--------------+--------+--------+----------+-------+
|          T001|  Laptop|       1|    1200.0| 1200.0|
|          T002|   Mouse|       2|      25.0|   50.0|
|          T003|Keyboard|       1|      45.0|   45.0|
|          T005|  Laptop|       1|    1200.0| 1200.0|
|          T006| Headset|       3|      40.0|  120.0|
|          T007|  Webcam|       1|      60.0|   60.0|
|          T010|   Phone|       2|     500.0| 1000.0|
+--------------+--------+--------+----------+-------+



## 7. Mengubah tipe data tanggal

Tugas: konversi `transaction_date` dari string menjadi tipe tanggal (`date`) agar siap digunakan untuk analisis.

In [7]:
from pyspark.sql.functions import to_date

# TODO 7: konversi transaction_date menjadi date
sales_df = sales_df.withColumn(
    "transaction_date",
    to_date(col("transaction_date"), "yyyy-MM-dd")
)

sales_df.printSchema()

root
 |-- transaction_id: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- status: string (nullable = true)
 |-- revenue: double (nullable = true)



## 8. Tahap Aggregate — ringkasan penjualan per kategori

Tugas: hitung total transaksi, total unit terjual, dan total revenue per kategori. Gunakan `groupBy()` dan `agg()`.

In [8]:
from pyspark.sql.functions import count, sum as spark_sum

# TODO 8: agregasi berdasarkan category
category_summary = (
    sales_df
    .groupBy("category")
    .agg(
        count("transaction_id").alias("total_transactions"),
        spark_sum("quantity").alias("total_units"),
        spark_sum("revenue").alias("total_revenue")
    )
)

category_summary.orderBy(col("total_revenue").desc()).show()

+-----------+------------------+-----------+-------------+
|   category|total_transactions|total_units|total_revenue|
+-----------+------------------+-----------+-------------+
|Electronics|                 3|          4|       3400.0|
|Accessories|                 4|          7|        275.0|
+-----------+------------------+-----------+-------------+



## 9. Agregasi berdasarkan tanggal

Tugas: hitung total revenue per hari.

In [9]:
# TODO 9: group by transaction_date dan sum revenue
daily_summary = (
    sales_df
    .groupBy("transaction_date")
    .agg(spark_sum("revenue").alias("daily_revenue"))
    .orderBy("transaction_date")
)

daily_summary.show()

+----------------+-------------+
|transaction_date|daily_revenue|
+----------------+-------------+
|      2026-10-20|       1295.0|
|      2026-10-21|       1380.0|
|      2026-10-22|       1000.0|
+----------------+-------------+



## 10. Data quality report

Buat ringkasan jumlah baris pada setiap tahap agar proses pipeline dapat diaudit.

In [10]:
# TODO 10: lengkapi dictionary ringkasan jumlah baris
quality_report = {
    "raw_rows": raw_df.count(),
    "duplicate_rows_removed": raw_df.count() - dedup_df.count(),
    "deduplicated_rows": dedup_df.count(),
    "invalid_rows_removed": dedup_df.count() - valid_df.count(),
    "valid_rows": valid_df.count(),
    "final_sales_rows": sales_df.count()
}

for stage, row_count in quality_report.items():
    print(f"{stage}: {row_count}")

raw_rows: 11
duplicate_rows_removed: 1
deduplicated_rows: 10
invalid_rows_removed: 3
valid_rows: 7
final_sales_rows: 7


## 11. Tahap Load — simpan output ke Parquet

Simpan data transaksi bersih dan dua ringkasan agregasi ke folder output terpisah. Gunakan `overwrite` supaya notebook dapat dijalankan ulang.

In [11]:
# TODO 11A: simpan sales_df ke output/clean_sales
(
    sales_df.write
    .format("parquet")
    .mode("overwrite")
    .save("output/clean_sales")
)

In [12]:
# TODO 11B: simpan category_summary ke output/category_summary
(
    category_summary.write
    .format("parquet")
    .mode("overwrite")
    .save("output/category_summary")
)

In [13]:
# TODO 11C: simpan daily_summary ke output/daily_summary
(
    daily_summary.write
    .format("parquet")
    .mode("overwrite")
    .save("output/daily_summary")
)

## 12. Verifikasi hasil

Baca kembali salah satu output Parquet untuk memastikan proses load berhasil.

In [14]:
loaded_summary = spark.read.parquet("output/category_summary")
loaded_summary.show()

+-----------+------------------+-----------+-------------+
|   category|total_transactions|total_units|total_revenue|
+-----------+------------------+-----------+-------------+
|Electronics|                 3|          4|       3400.0|
|Accessories|                 4|          7|        275.0|
+-----------+------------------+-----------+-------------+



## 13. Pertanyaan interpretasi

Jawab berdasarkan hasil yang diperoleh:
1. Berapa banyak baris yang dihapus oleh proses deduplikasi?
2. Mengapa transaksi `CANCELLED` tidak dihitung sebagai revenue?
3. Mengapa quantity bernilai 0 dan unit_price bernilai null tidak dipertahankan?
4. Kategori mana yang menghasilkan total revenue terbesar?
5. Apa keuntungan menyimpan output analitik ke Parquet?
6. Jika data mentah bertambah setiap hari, bagaimana pipeline ini dapat dijadwalkan dan dibuat lebih mudah dipantau?

In [ ]:
# Jawaban 1: Satu baris duplikat dihapus, yaitu salinan kedua transaksi T007.
# Jawaban 2: Transaksi CANCELLED tidak merepresentasikan penjualan selesai, sehingga tidak boleh menambah revenue.
# Jawaban 3: Quantity 0 tidak menghasilkan penjualan, sedangkan unit_price null membuat revenue tidak dapat dihitung secara andal.
# Jawaban 4: Electronics menghasilkan total revenue terbesar, yaitu USD 3,400.
# Jawaban 5: Parquet bersifat columnar, terkompresi, menyimpan schema, dan mendukung pembacaan kolom serta filter secara efisien.
# Jawaban 6: Pipeline dapat dijadwalkan dengan Airflow, dibuat idempotent dan terpartisi per tanggal, serta dipantau melalui log, metrik kualitas, retry, dan alert.

## Checklist sebelum dikumpulkan

- [x] Semua bagian TODO sudah dilengkapi.
- [x] Jumlah baris pada setiap tahap sudah diperiksa.
- [x] Validasi dan deduplikasi bekerja sesuai aturan bisnis.
- [x] Output transaksi bersih dan agregasi berhasil disimpan ke Parquet.
- [x] Pertanyaan interpretasi telah dijawab.
- [x] Notebook disimpan dengan output yang terlihat.

**Pengumpulan:** kumpulkan file `.ipynb` yang sudah dikerjakan.